# M2 · YOLOv11n + EMAR 残差注意力@颈部P3（Colab，Drive 持久化版）

对 M1/M1' 两轮失败的根因修正：**残差封装 y = x + EMA(x)**。核心逻辑已通过本地实跑验证。

**本版新增：结果写入 Google Drive——训练中途没额度/断线不丢进度，重新运行本笔记本会自动从 last.pt 续训。**

操作：T4 → 全部运行 → 中途授权访问 Google Drive → 约 2 小时 → 下载 results.zip。

In [ ]:
!nvidia-smi

In [ ]:
# 下载数据集（同前）
API_KEY = 'YOUR_ROBOFLOW_API_KEY'

import json, glob, zipfile, urllib.request, subprocess

meta = json.load(urllib.request.urlopen(
    f'https://api.roboflow.com/km-sd0ce/pig-behavior-wlvku/1/yolov8?api_key={API_KEY}'))
subprocess.run(['curl', '-sL', '-o', '/content/dataset.zip', meta['export']['link']], check=True)
with zipfile.ZipFile('/content/dataset.zip') as z:
    z.extractall('/content/dataset')
DATA_YAML = glob.glob('/content/dataset/**/data.yaml', recursive=True)[0]
print('data.yaml:', DATA_YAML)

In [ ]:
!pip install -q ultralytics
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
# EMA/EMAR 定义 + 注册 + 改进 yaml + 权重重映射 + 结构自检（本地已验证同版代码）
import re, torch
from torch import nn
import ultralytics.nn.tasks as tasks

class EMA(nn.Module):
    """Efficient Multi-Scale Attention (ICASSP 2023)，输入输出通道数不变。"""
    def __init__(self, channels, factor=8):
        super().__init__()
        self.groups = factor
        assert channels // self.groups > 0
        self.softmax = nn.Softmax(-1)
        self.agp = nn.AdaptiveAvgPool2d((1, 1))
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.gn = nn.GroupNorm(channels // self.groups, channels // self.groups)
        self.conv1x1 = nn.Conv2d(channels // self.groups, channels // self.groups, 1)
        self.conv3x3 = nn.Conv2d(channels // self.groups, channels // self.groups, 3, padding=1)

    def forward(self, x):
        b, c, h, w = x.size()
        group_x = x.reshape(b * self.groups, -1, h, w)
        x_h = self.pool_h(group_x)
        x_w = self.pool_w(group_x).permute(0, 1, 3, 2)
        hw = self.conv1x1(torch.cat([x_h, x_w], dim=2))
        x_h, x_w = torch.split(hw, [h, w], dim=2)
        x1 = self.gn(group_x * x_h.sigmoid() * x_w.permute(0, 1, 3, 2).sigmoid())
        x2 = self.conv3x3(group_x)
        x11 = self.softmax(self.agp(x1).reshape(b * self.groups, -1, 1).permute(0, 2, 1))
        x12 = x2.reshape(b * self.groups, c // self.groups, -1)
        x21 = self.softmax(self.agp(x2).reshape(b * self.groups, -1, 1).permute(0, 2, 1))
        x22 = x1.reshape(b * self.groups, c // self.groups, -1)
        weights = (torch.matmul(x11, x12) + torch.matmul(x21, x22)).reshape(b * self.groups, 1, h, w)
        return (group_x * weights.sigmoid()).reshape(b, c, h, w)

class EMAR(nn.Module):
    """EMA 的残差封装：y = x + EMA(x)，插入预训练网络不破坏原特征分布。"""
    def __init__(self, channels):
        super().__init__()
        self.ema = EMA(channels)

    def forward(self, x):
        return x + self.ema(x)

tasks.EMA = EMA
tasks.EMAR = EMAR

YAML_TEXT = r'''
nc: 10
scales:
  n: [0.50, 0.25, 1024]
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]
  - [-1, 1, EMAR, [64]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]
  - [[17, 20, 23], 1, Detect, [nc]]
'''
with open('/content/yolo11-emar-n.yaml', 'w') as f:
    f.write(YAML_TEXT.strip() + '\n')

from ultralytics import YOLO
from ultralytics.utils.downloads import attempt_download_asset
model = YOLO('/content/yolo11-emar-n.yaml')

ckpt = torch.load(attempt_download_asset('yolo11n.pt'), map_location='cpu', weights_only=False)
sd = ckpt['model'].state_dict() if 'model' in ckpt else ckpt
remapped = {}
for k, v in sd.items():
    m = re.match(r'model\.(\d+)\.', k)
    if m and 17 <= int(m.group(1)) <= 23:
        k = k.replace(f'model.{m.group(1)}.', f'model.{int(m.group(1))+1}.', 1)
    remapped[k] = v
model_sd = model.model.state_dict()
remapped = {k: v for k, v in remapped.items() if k in model_sd and model_sd[k].shape == v.shape}
missing, unexpected = model.model.load_state_dict(remapped, strict=False)
print(f'权重迁移：成功 {len(remapped)} 键；缺失 {len(missing)} 键（应为 57：EMAR 6 + Detect 分类头 51）；多余 {len(unexpected)} 键')

with torch.no_grad():
    _ = model.model.eval()(torch.zeros(1, 3, 640, 640))
print('结构自检通过')

In [ ]:
# 训练 M2（Drive 持久化 + 自动断点续训）
from google.colab import drive
drive.mount('/content/drive')  # 中途弹出授权链接，点按提示授权即可

import os
DRIVE_RESULTS = '/content/drive/MyDrive/pig-results'
LAST_PT = f'{DRIVE_RESULTS}/m2-emar/weights/last.pt'

if os.path.exists(LAST_PT):
    print('检测到上次未完成的训练，从 last.pt 断点续训')
    model = YOLO(LAST_PT)
    model.train(resume=True)
else:
    print('全新训练（结果实时写入 Drive，中断不丢进度）')
    model.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16,
                device=0, project=DRIVE_RESULTS, name='m2-emar')

In [ ]:
# 评估 + 保存指标（Drive 路径）
import json, os

DRIVE_RESULTS = '/content/drive/MyDrive/pig-results'
metrics = model.val()
summary = {
    'mAP50': round(float(metrics.box.map50), 4),
    'mAP50-95': round(float(metrics.box.map), 4),
    'precision': round(float(metrics.box.mp), 4),
    'recall': round(float(metrics.box.mr), 4),
}
os.makedirs(f'{DRIVE_RESULTS}/m2-emar', exist_ok=True)
with open(f'{DRIVE_RESULTS}/m2-emar/metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(summary)
print('对照基线: mAP50=0.5706 | M1: 0.5411(否决) | 判决线: >0.581 有效')

In [ ]:
# 打包 Drive 上的结果并下载
import shutil
from google.colab import files

shutil.make_archive('/content/results', 'zip', '/content/drive/MyDrive/pig-results')
files.download('/content/results.zip')